# Demo 12a — What an MLP cannot do

**Class 12 · M1–M2** (slides: `slides/12_neural_networks.md`)

A multilayer perceptron (Class 05) reads a fixed-length vector and treats every input as unrelated to its neighbours.

* **M1:** parameters of the first dense layer for images of growing size.
* **M2:** the same MLP on the digits, on digits with the pixels shuffled by one fixed permutation, and on shifted digits.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import keras
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split

from faa import demo, neural, plotlib
from faa.demo import show, slide

demo.setup("12")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0

## Slide: Parameters of the first layer (M1)

One dense layer with $h$ units on an image of $H \times W \times C$ numbers has $(HWC + 1)\,h$ parameters.

In [ ]:
rows = []
for name, (H, W, C) in {"digits": (8, 8, 1), "MNIST": (28, 28, 1), "photo 224": (224, 224, 3)}.items():
    for h in (64, 1000):
        rows.append(
            {
                "image": name,
                "H x W x C": f"{H}x{W}x{C}",
                "inputs": H * W * C,
                "units h": h,
                "parameters": (H * W * C + 1) * h,
            }
        )
slide("Parameters of the first dense layer")
show(pl.DataFrame(rows))

f, ax = plotlib.fig("col", 1.7)
sizes = np.array([8, 16, 28, 64, 128, 224])
ax.loglog(sizes, (sizes**2 + 1) * 64, "o-", color=BLUE, label="dense, 64 units")
ax.loglog(sizes, np.full(len(sizes), 3 * 3 * 1 * 64 + 64), "s-", color=ORANGE, label="conv 3x3, 64 filters")
ax.set_xlabel("image side $H = W$")
ax.set_ylabel("parameters")
ax.legend(frameon=False)
plotlib.save(f, "m1_params_image")

## Slide: An MLP on the digits (M2)

The 1 797 digits as 8×8 images. A 64–64–32–10 MLP, 60 % train, 20 % validation, 20 % test; five seeds.

In [ ]:
X, y = neural.load_digits_images()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=SEED, stratify=y)
Xva, Xte, yva, yte = train_test_split(Xte, yte, test_size=0.5, random_state=SEED, stratify=yte)


def fit_eval(make, Xa, ya, tests: dict, seeds=range(3), epochs=30):
    """Train `make(seed)` on (Xa, ya); return {test name: accuracies over the seeds}."""
    res = {k: [] for k in tests}
    for s in seeds:
        m = make(s)
        m.fit(Xa, ya, epochs=epochs, batch_size=32, validation_data=(Xva, yva), verbose=0)
        for k, (Xt, yt) in tests.items():
            res[k].append(m.evaluate(Xt, yt, verbose=0)[1])
    return res


def mlp(s):
    return neural.mlp_model((8, 8, 1), hidden=(64, 32), seed=s)


res = fit_eval(mlp, Xtr, ytr, {"test": (Xte, yte)})
slide("An MLP on the digits")
print(
    f"parameters: {neural.n_params(mlp(0))}; test accuracy {np.mean(res['test']):.3f} +- {np.std(res['test']):.3f} over 3 seeds"
)

## Slide: Shuffle the pixels, nothing changes (M2)

One fixed random permutation of the 64 pixel positions is applied to every image. A person cannot read the digits any more;
the MLP does not care, because it never used the neighbourhood.

In [ ]:
rng = np.random.default_rng(SEED)
perm = rng.permutation(64)


def shuf(A):
    return A.reshape(len(A), 64)[:, perm].reshape(-1, 8, 8, 1)


res_p = fit_eval(mlp, shuf(Xtr), ytr, {"test": (shuf(Xte), yte)})
slide("Shuffled pixels")
print(f"original {np.mean(res['test']):.3f}, shuffled {np.mean(res_p['test']):.3f}")

f, ax = plt.subplots(1, 2, figsize=(plotlib.COL, 1.0))
for a, img, t in zip(ax, (Xtr[0], shuf(Xtr[:1])[0]), ("original", "shuffled")):
    a.imshow(img[..., 0], cmap="gray_r")
    a.set_title(t)
    a.axis("off")
plotlib.save(f, "m2_shuffled")

## Slide: Shift the digit (M2)

Train on the centred digits, test on digits moved by up to $s$ pixels. The sweep is the failure case: the accuracy falls with $s$.

In [ ]:
shifts = (0, 1, 2)
tests = {s: (neural.random_shift(Xte, s, seed=1), yte) for s in shifts}
res_s = fit_eval(mlp, Xtr, ytr, tests)
rows = [{"max shift s": s, "accuracy": np.mean(res_s[s]), "std": np.std(res_s[s])} for s in shifts]
slide("Shifted digits, MLP")
show(pl.DataFrame(rows))
f, ax = plotlib.fig("col", 1.5)
ax.bar(
    [str(r["max shift s"]) for r in rows],
    [r["accuracy"] for r in rows],
    yerr=[r["std"] for r in rows],
    color=[BLUE, ORANGE, RED],
)
for i, r in enumerate(rows):
    ax.text(i, r["accuracy"] + 0.04, f"{r['accuracy']:.2f}", ha="center", fontsize=6)
ax.set_xlabel("maximum shift $s$ (pixels)")
ax.set_ylabel("test accuracy")
ax.set_ylim(0, 1.1)
plotlib.save(f, "m2_shift")

## Failure case: an order task on a flattened sequence

The MLP also flattens a sequence, so it can learn the order task of `faa.neural.order_task`, but each position has its own weights:
it needs more examples as the sequence grows.

In [ ]:
rows = []
for length in (6, 12, 24):
    xs, ys = neural.order_task(4000, length, seed=1)
    xt, yt = neural.order_task(1000, length, seed=2)
    accs = []
    for s in range(3):
        keras.utils.set_random_seed(s)
        m = keras.Sequential(
            [
                keras.layers.Input((length, 2)),
                keras.layers.Flatten(),
                keras.layers.Dense(64, activation="relu"),
                keras.layers.Dense(2, activation="softmax"),
            ]
        )
        neural.compile_clf(m)
        m.fit(xs, ys, epochs=25, batch_size=64, verbose=0)
        accs.append(m.evaluate(xt, yt, verbose=0)[1])
    rows.append({"length": length, "parameters": neural.n_params(m), "accuracy": np.mean(accs)})
slide("MLP on the order task")
show(pl.DataFrame(rows))